# PyTorch Geometric Building Graph Representation (BGR) Classification

This notebook demonstrates Building Graph Representation (BGR) classification using
**topologic_fast** and PyTorch Geometric. BGR is a graph-based representation of
building topology that captures spatial relationships between building elements.

## What is BGR?

Building Graph Representation converts a 3D building model into a graph where:
- **Nodes** represent spatial units (rooms, cells, zones)
- **Edges** represent adjacency relationships (shared walls, connections)
- **Node features** encode spatial properties (area, volume, position)
- **Edge features** encode interface properties (shared area, permeability)

## Prerequisites

```bash
pip install topologic_fast torch torch_geometric plotly pandas scikit-learn
```

## Import Libraries

In [ ]:
# Core imports
import topologic_fast as tf
import numpy as np
import pandas as pd
from pathlib import Path

# PyTorch and PyG
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv, SAGEConv, global_mean_pool, global_max_pool
from torch_geometric.nn import BatchNorm

# Visualization
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# ML utilities
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import torch_geometric
print(f"topologic_fast loaded")
print(f"PyTorch: {torch.__version__}")
print(f"PyG: {torch_geometric.__version__}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## Building Graph Representation (BGR) Extraction

Extract comprehensive graph features from building topologies.

In [ ]:
class BuildingGraphExtractor:
    """
    Extracts Building Graph Representation from topologic_fast topologies.
    
    Node features:
    - Centroid coordinates (x, y, z)
    - Cell volume
    - Cell surface area
    - Degree (number of adjacent cells)
    - Floor level (estimated from z)
    - Compactness ratio
    
    Edge features (optional):
    - Shared face area
    - Interface direction
    """
    
    def __init__(self, tolerance=0.001):
        self.tolerance = tolerance
        
    def extract(self, topology, label=None):
        """
        Extract BGR from a CellComplex or similar topology.
        
        Returns:
            PyG Data object with node and edge features
        """
        # Create dual graph
        graph = tf.Graph.ByTopology(topology, direct=True, tolerance=self.tolerance)
        
        vertices = graph.Vertices()
        edges = graph.Edges()
        adj_list = graph.AdjacencyList()
        
        num_nodes = graph.Order()
        if num_nodes == 0:
            return None
        
        # Extract node features
        node_features = self._extract_node_features(vertices, adj_list)
        
        # Build edge index
        edge_index = self._build_edge_index(adj_list, num_nodes)
        
        # Create PyG Data
        x = torch.tensor(node_features, dtype=torch.float32)
        
        data = Data(x=x, edge_index=edge_index)
        
        if label is not None:
            data.y = torch.tensor([label], dtype=torch.long)
        
        # Add graph-level features
        data.num_cells = num_nodes
        data.num_adjacencies = graph.Size()
        data.density = graph.Density()
        
        return data
    
    def _extract_node_features(self, vertices, adj_list):
        """
        Extract features for each node (cell centroid).
        """
        features = []
        
        # Compute bounds for normalization
        coords = [v.Coordinates() for v in vertices]
        if not coords:
            return [[0.0] * 8]  # Empty fallback
            
        xs = [c[0] for c in coords]
        ys = [c[1] for c in coords]
        zs = [c[2] for c in coords]
        
        x_range = max(xs) - min(xs) + 1e-6
        y_range = max(ys) - min(ys) + 1e-6
        z_range = max(zs) - min(zs) + 1e-6
        
        for i, v in enumerate(vertices):
            x, y, z = v.Coordinates()
            
            # Normalized position
            x_norm = (x - min(xs)) / x_range
            y_norm = (y - min(ys)) / y_range
            z_norm = (z - min(zs)) / z_range
            
            # Degree
            degree = len(adj_list[i]) if i < len(adj_list) else 0
            
            # Estimated floor level
            floor_height = 3.0
            floor_level = int(z / floor_height)
            
            # Position in building (corner=0, edge=1, interior=2)
            is_corner = 1.0 if (x_norm < 0.1 or x_norm > 0.9) and (y_norm < 0.1 or y_norm > 0.9) else 0.0
            is_edge = 1.0 if (x_norm < 0.1 or x_norm > 0.9 or y_norm < 0.1 or y_norm > 0.9) else 0.0
            
            features.append([
                x_norm,
                y_norm,
                z_norm,
                float(degree),
                float(floor_level),
                is_corner,
                is_edge,
                float(degree > 4)  # High connectivity indicator
            ])
        
        return features
    
    def _build_edge_index(self, adj_list, num_nodes):
        """
        Build edge index tensor from adjacency list.
        """
        src_nodes = []
        dst_nodes = []
        
        for i, neighbors in enumerate(adj_list):
            for j in neighbors:
                src_nodes.append(i)
                dst_nodes.append(j)
        
        if len(src_nodes) == 0:
            # Self-loops for isolated graphs
            src_nodes = list(range(num_nodes))
            dst_nodes = list(range(num_nodes))
        
        return torch.tensor([src_nodes, dst_nodes], dtype=torch.long)

# Test BGR extraction
extractor = BuildingGraphExtractor()

# Create test building
cells = []
for i in range(3):
    for j in range(3):
        for k in range(2):
            cells.append(tf.Cell.Box(i, j, k*3, 1, 1, 3))

test_cc = tf.CellComplex.ByCells(cells)
test_data = extractor.extract(test_cc, label=0)

print(f"Test BGR:")
print(f"  Nodes: {test_data.x.shape}")
print(f"  Edges: {test_data.edge_index.shape}")
print(f"  Features per node: {test_data.x.shape[1]}")

## Define Building Types

Create different architectural building types for classification.

In [ ]:
class BuildingGenerator:
    """
    Generates building topologies for different architectural types.
    
    Building types:
    0: Slab - Wide, flat, single block
    1: Tower - Tall, narrow, vertical stack
    2: Courtyard - U or L shaped with interior space
    3: Atrium - Central void with surrounding cells
    4: Cluster - Irregular grouping of cells
    """
    
    def __init__(self, floor_height=3.0):
        self.floor_height = floor_height
    
    def generate(self, building_type, **kwargs):
        """
        Generate a building CellComplex.
        """
        if building_type == 0 or building_type == 'slab':
            return self._slab(**kwargs)
        elif building_type == 1 or building_type == 'tower':
            return self._tower(**kwargs)
        elif building_type == 2 or building_type == 'courtyard':
            return self._courtyard(**kwargs)
        elif building_type == 3 or building_type == 'atrium':
            return self._atrium(**kwargs)
        elif building_type == 4 or building_type == 'cluster':
            return self._cluster(**kwargs)
        else:
            return self._slab(**kwargs)
    
    def _slab(self, width=5, length=3, floors=2):
        """Wide, flat building."""
        cells = []
        for i in range(width):
            for j in range(length):
                for f in range(floors):
                    z = f * self.floor_height
                    cells.append(tf.Cell.Box(i, j, z, 1, 1, self.floor_height))
        return tf.CellComplex.ByCells(cells) if cells else None
    
    def _tower(self, base_width=2, base_length=2, floors=6):
        """Tall, narrow building."""
        cells = []
        for i in range(base_width):
            for j in range(base_length):
                for f in range(floors):
                    z = f * self.floor_height
                    cells.append(tf.Cell.Box(i, j, z, 1, 1, self.floor_height))
        return tf.CellComplex.ByCells(cells) if cells else None
    
    def _courtyard(self, outer_size=5, inner_size=2, floors=3):
        """Building with interior courtyard."""
        cells = []
        inner_start = (outer_size - inner_size) // 2
        inner_end = inner_start + inner_size
        
        for i in range(outer_size):
            for j in range(outer_size):
                # Skip interior courtyard
                if inner_start <= i < inner_end and inner_start <= j < inner_end:
                    continue
                for f in range(floors):
                    z = f * self.floor_height
                    cells.append(tf.Cell.Box(i, j, z, 1, 1, self.floor_height))
        return tf.CellComplex.ByCells(cells) if cells else None
    
    def _atrium(self, size=4, floors=4, atrium_floors=3):
        """Building with central atrium void."""
        cells = []
        center = size // 2
        
        for i in range(size):
            for j in range(size):
                for f in range(floors):
                    # Create void in center for lower floors
                    if i == center and j == center and f < atrium_floors:
                        continue
                    z = f * self.floor_height
                    cells.append(tf.Cell.Box(i, j, z, 1, 1, self.floor_height))
        return tf.CellComplex.ByCells(cells) if cells else None
    
    def _cluster(self, num_blocks=5, max_size=3, max_floors=3):
        """Irregular cluster of building blocks."""
        cells = []
        np.random.seed(None)  # Random each time
        
        offset_x = 0
        for b in range(num_blocks):
            w = np.random.randint(1, max_size + 1)
            l = np.random.randint(1, max_size + 1)
            h = np.random.randint(1, max_floors + 1)
            offset_y = np.random.randint(0, 3)
            
            for i in range(w):
                for j in range(l):
                    for f in range(h):
                        z = f * self.floor_height
                        cells.append(tf.Cell.Box(offset_x + i, offset_y + j, z, 1, 1, self.floor_height))
            
            offset_x += w + np.random.randint(0, 2)
        
        return tf.CellComplex.ByCells(cells) if cells else None

# Test building generation
generator = BuildingGenerator()

for i, name in enumerate(['Slab', 'Tower', 'Courtyard', 'Atrium', 'Cluster']):
    building = generator.generate(i)
    if building:
        graph = tf.Graph.ByTopology(building, direct=True)
        print(f"{name}: {graph.Order()} cells, {graph.Size()} adjacencies")

## Generate Training Dataset

In [ ]:
def generate_bgr_dataset(samples_per_class=25):
    """
    Generate a dataset of building graphs with varied parameters.
    """
    generator = BuildingGenerator()
    extractor = BuildingGraphExtractor()
    
    data_list = []
    np.random.seed(42)
    
    # Class configurations with parameter ranges
    class_configs = {
        0: ('slab', {
            'width': (4, 8),
            'length': (2, 4),
            'floors': (1, 3)
        }),
        1: ('tower', {
            'base_width': (1, 3),
            'base_length': (1, 3),
            'floors': (4, 8)
        }),
        2: ('courtyard', {
            'outer_size': (4, 6),
            'inner_size': (1, 2),
            'floors': (2, 4)
        }),
        3: ('atrium', {
            'size': (3, 5),
            'floors': (3, 5),
            'atrium_floors': (2, 3)
        }),
        4: ('cluster', {
            'num_blocks': (3, 6),
            'max_size': (2, 4),
            'max_floors': (2, 4)
        })
    }
    
    class_names = ['Slab', 'Tower', 'Courtyard', 'Atrium', 'Cluster']
    
    for label, (building_type, param_ranges) in class_configs.items():
        print(f"Generating {class_names[label]} buildings...")
        
        for _ in range(samples_per_class):
            # Random parameters within range
            params = {}
            for param, (low, high) in param_ranges.items():
                params[param] = np.random.randint(low, high + 1)
            
            try:
                topology = generator.generate(building_type, **params)
                if topology is not None:
                    data = extractor.extract(topology, label=label)
                    if data is not None and data.x.shape[0] > 0:
                        data_list.append(data)
            except Exception as e:
                pass
    
    return data_list, class_names

# Generate dataset
print("Generating BGR dataset...\n")
dataset, class_names = generate_bgr_dataset(samples_per_class=25)
print(f"\nTotal graphs: {len(dataset)}")

# Statistics
labels = [d.y.item() for d in dataset]
print(f"\nClass distribution:")
for i, name in enumerate(class_names):
    count = labels.count(i)
    print(f"  {name}: {count}")

## Define BGR Classification Model

In [ ]:
class BGRClassifier(nn.Module):
    """
    Graph Neural Network for Building Graph Representation classification.
    
    Architecture:
    - Multiple GraphSAGE layers with batch normalization
    - Skip connections for better gradient flow
    - Dual pooling (mean + max) for richer graph representation
    - MLP classifier head
    """
    
    def __init__(self, in_channels, hidden_channels, num_classes, num_layers=3, dropout=0.3):
        super(BGRClassifier, self).__init__()
        
        self.num_layers = num_layers
        self.dropout = dropout
        
        # Graph convolutions
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        
        # Input projection
        self.input_proj = nn.Linear(in_channels, hidden_channels)
        
        for _ in range(num_layers):
            self.convs.append(SAGEConv(hidden_channels, hidden_channels))
            self.bns.append(nn.BatchNorm1d(hidden_channels))
        
        # Dual pooling -> 2x hidden channels
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(hidden_channels * 2, hidden_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels, hidden_channels // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels // 2, num_classes)
        )
        
    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        
        # Input projection
        x = F.relu(self.input_proj(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        
        # Graph convolutions with skip connections
        for i, (conv, bn) in enumerate(zip(self.convs, self.bns)):
            x_in = x
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = x + x_in  # Skip connection
        
        # Dual pooling
        x_mean = global_mean_pool(x, batch)
        x_max = global_max_pool(x, batch)
        x = torch.cat([x_mean, x_max], dim=1)
        
        return self.classifier(x)

# Initialize model
in_channels = 8  # From BGRExtractor
hidden_channels = 64
num_classes = 5

model = BGRClassifier(in_channels, hidden_channels, num_classes, num_layers=3).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(model)

## Train the Model

In [ ]:
# Split data
labels = [d.y.item() for d in dataset]
indices = list(range(len(dataset)))

train_idx, temp_idx = train_test_split(indices, test_size=0.2, random_state=42, stratify=labels)
temp_labels = [labels[i] for i in temp_idx]
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42, stratify=temp_labels)

train_dataset = [dataset[i] for i in train_idx]
val_dataset = [dataset[i] for i in val_idx]
test_dataset = [dataset[i] for i in test_idx]

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)
test_loader = DataLoader(test_dataset, batch_size=16)

# Training setup
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

# Training loop
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0
best_model = None

print("\nTraining...")
for epoch in range(100):
    # Train
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0
    
    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data)
        loss = criterion(out, data.y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        train_loss += loss.item() * data.num_graphs
        pred = out.argmax(dim=1)
        train_correct += (pred == data.y).sum().item()
        train_total += data.num_graphs
    
    train_loss /= train_total
    train_acc = train_correct / train_total
    
    # Validate
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for data in val_loader:
            data = data.to(device)
            out = model(data)
            loss = criterion(out, data.y)
            
            val_loss += loss.item() * data.num_graphs
            pred = out.argmax(dim=1)
            val_correct += (pred == data.y).sum().item()
            val_total += data.num_graphs
    
    val_loss /= val_total
    val_acc = val_correct / val_total
    
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model = model.state_dict().copy()
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Train Acc={train_acc:.4f}, "
              f"Val Loss={val_loss:.4f}, Val Acc={val_acc:.4f}")

# Load best model
if best_model:
    model.load_state_dict(best_model)
print(f"\nBest validation accuracy: {best_val_acc:.4f}")

## Evaluate and Visualize Results

In [ ]:
# Test evaluation
model.eval()
test_preds = []
test_labels_list = []

with torch.no_grad():
    for data in test_loader:
        data = data.to(device)
        out = model(data)
        pred = out.argmax(dim=1)
        test_preds.extend(pred.cpu().tolist())
        test_labels_list.extend(data.y.cpu().tolist())

test_acc = accuracy_score(test_labels_list, test_preds)
print(f"Test Accuracy: {test_acc:.4f}")
print("\nClassification Report:")
print(classification_report(test_labels_list, test_preds, target_names=class_names, zero_division=0))

In [ ]:
# Training curves
epochs = list(range(1, len(history['train_loss']) + 1))

fig = make_subplots(rows=1, cols=2, subplot_titles=('Loss', 'Accuracy'))

fig.add_trace(go.Scatter(x=epochs, y=history['train_loss'], name='Train Loss', line=dict(color='blue')), row=1, col=1)
fig.add_trace(go.Scatter(x=epochs, y=history['val_loss'], name='Val Loss', line=dict(color='orange')), row=1, col=1)
fig.add_trace(go.Scatter(x=epochs, y=history['train_acc'], name='Train Acc', line=dict(color='blue')), row=1, col=2)
fig.add_trace(go.Scatter(x=epochs, y=history['val_acc'], name='Val Acc', line=dict(color='orange')), row=1, col=2)

fig.update_layout(title='BGR Classification Training', height=400, width=900)
fig.show()

In [ ]:
# Confusion matrix
cm = confusion_matrix(test_labels_list, test_preds)

fig = go.Figure(data=go.Heatmap(
    z=cm,
    x=class_names,
    y=class_names,
    colorscale='Blues',
    text=cm,
    texttemplate='%{text}',
    textfont={'size': 14}
))

fig.update_layout(
    title='BGR Classification - Confusion Matrix',
    xaxis_title='Predicted',
    yaxis_title='Actual',
    width=600, height=500
)
fig.show()

## Visualize Building Graphs

In [ ]:
def visualize_building_bgr(topology, title):
    """
    Visualize building as a 3D graph.
    """
    graph = tf.Graph.ByTopology(topology, direct=True, tolerance=0.001)
    vertices = graph.Vertices()
    edges = graph.Edges()
    adj_list = graph.AdjacencyList()
    
    # Node positions and colors
    xs = [v.X() for v in vertices]
    ys = [v.Y() for v in vertices]
    zs = [v.Z() for v in vertices]
    degrees = [len(adj_list[i]) if i < len(adj_list) else 0 for i in range(len(vertices))]
    
    # Edge lines
    xe, ye, ze = [], [], []
    for edge in edges:
        s, e = edge.StartVertex(), edge.EndVertex()
        xe.extend([s.X(), e.X(), None])
        ye.extend([s.Y(), e.Y(), None])
        ze.extend([s.Z(), e.Z(), None])
    
    fig = go.Figure()
    
    # Edges
    fig.add_trace(go.Scatter3d(
        x=xe, y=ye, z=ze,
        mode='lines',
        line=dict(color='lightgray', width=1),
        name='Adjacencies'
    ))
    
    # Nodes
    fig.add_trace(go.Scatter3d(
        x=xs, y=ys, z=zs,
        mode='markers',
        marker=dict(
            size=8,
            color=degrees,
            colorscale='Viridis',
            colorbar=dict(title='Degree'),
            opacity=0.9
        ),
        name='Cells'
    ))
    
    fig.update_layout(
        title=f"{title} ({graph.Order()} cells, {graph.Size()} adjacencies)",
        scene=dict(aspectmode='data'),
        width=550, height=450,
        margin=dict(l=0, r=0, t=40, b=0)
    )
    
    return fig

# Visualize each building type
generator = BuildingGenerator()

for i, name in enumerate(class_names):
    try:
        building = generator.generate(i)
        if building:
            fig = visualize_building_bgr(building, name)
            fig.show()
    except Exception as e:
        print(f"Could not visualize {name}: {e}")

## Notes on BGR Classification with topologic_fast

### Key Concepts

**Building Graph Representation (BGR)**:
- Converts 3D building geometry to graph structure
- Captures spatial relationships between building elements
- Enables machine learning on architectural designs

### API Usage

| Operation | topologic_fast |
|-----------|----------------|
| Create cells | `tf.Cell.Box(x, y, z, w, l, h)` |
| Create building | `tf.CellComplex.ByCells(cells)` |
| Extract graph | `tf.Graph.ByTopology(cc, direct=True)` |
| Get adjacency | `graph.AdjacencyList()` |
| Get vertices | `graph.Vertices()` |

### Features Not Yet Implemented

```python
# NOT YET AVAILABLE - Use manual extraction shown above:
# PyG.DatasetByCSVPath() - Load from CSV
# PyG.ByCSVPath() - Create PyG wrapper
# Direct cell volume/area extraction from graph vertices
```

### Applications

- Building type classification
- Energy performance prediction
- Spatial quality assessment
- Design similarity search

In [ ]:
# Clean up
tf.clear_store()
print("Topology store cleared.")